# S2.T3.1 · Task 2 — Knowledge tracing vs. Elo baseline

Held-out **replay** of (item_id, correct) rows. We report:

1. **AUC** of *next-response* correctness: before each label, we output a probability (BKT from P(L) for the item’s subskill; Elo from σ(θ−b) with online θ).
2. **Top-1** hit rate for **next-item** choice from a pool of 5 (true next + 4 decoys), comparing **BKT (max response entropy / information)** vs **Elo (closest p to 0.5)**.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path(os.getcwd())
if (ROOT / "tutor").is_dir() and str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
from sklearn.metrics import roc_auc_score, brier_score_loss

from tutor.adaptive import (
    BKTModel,
    EloBaseline,
    generate_bkt_synthetic_sessions,
    next_item_top1_hits,
)

## Data
Synthetic BKT-consistent sessions (hidden generative BKT; train split fits **new** BKT and item difficulties for Elo). In your defense you can replace this with a CSV replay from the tutor or Common Core logs, keeping the same `item_skill` index.

In [ ]:
N_SESSIONS = 200
LEN = 32
N_ITEMS = 12
N_SK = 5
SEED = 42
RNG = np.random.default_rng(SEED)

all_sessions, _true_gen, item_skill = generate_bkt_synthetic_sessions(
    N_SESSIONS, LEN, N_ITEMS, N_SK, random_state=SEED
)
n_train = int(0.8 * N_SESSIONS)
idx = RNG.permutation(N_SESSIONS)
tr = [all_sessions[i] for i in idx[:n_train]]
te = [all_sessions[i] for i in idx[n_train:]]
len(tr), len(te), list(item_skill)

## Fit BKT and Elo on train; evaluate AUC on test

In [ ]:
bkt = BKTModel.fit(N_SK, tr, item_skill, random_state=SEED + 1)
elob = EloBaseline.from_train(tr, N_ITEMS)

yt, ys = BKTModel.extract_predict_sample_pairs(bkt, te, item_skill)
y2, s2 = EloBaseline.extract_elo_auc(elob, te)

def _auc(yt, ys):
    if len(np.unique(yt)) < 2:
        return float("nan")
    return roc_auc_score(yt, ys)

print("BKT  AUC (next-response):", _auc(yt, ys))
print("Elo  AUC (next-response):", _auc(y2, s2))
print("BKT  Brier:", brier_score_loss(yt, np.clip(ys, 1e-7, 1.0 - 1e-7)))
print("Elo  Brier:", brier_score_loss(y2, np.clip(s2, 1e-7, 1.0 - 1e-7)))

## Next-item selection (Top-1 in pool of 5)
Policies differ: BKT picks the item with **highest Bernoulli entropy** (uncertainty) under current beliefs; Elo picks where **p(correct) is closest to 0.5** given θ, b. Neither is the teacher that generated the data — expect modest hit rates; the point is a **comparative** read.

In [ ]:
hb, he = next_item_top1_hits(
    bkt, elob, te, item_skill, N_ITEMS, pool_size=5, random_state=SEED + 2
)
print(f"BKT  Top-1 (next in pool): {100 * hb:.1f}%")
print(f"Elo  Top-1 (next in pool): {100 * he:.1f}%")

## Fitted BKT parameters (illustration)
`p_t`, `p_g`, `p_s` shared; `p_L0` per subskill (counting, number_sense, …).

In [ ]:
print("p_L0 per skill:", bkt.p_L0)
print("p_t, p_g, p_s:", bkt.p_t, bkt.p_g, bkt.p_s)
print("first 4 item b (Elo):", elob.b_item[:4].round(3))